In [1]:
import qsharp
import random
import json
import time
from diskcache import Cache
from matplotlib import pyplot as plt


cache = Cache("~/quant-arith-cache/re-constant-adder-tmp")
qsharp.init(project_root="../")

@cache.memoize()
def estimate_resources_constant_adder(op, n):
    est = qsharp.estimate(f"EstimateUtils.RunConstantAdder({n},{op})")
    return json.dumps(est)    

In [ ]:
# These all are in-place adders modulo 2^n.
ops = [
  "Std.Arithmetic.IncByLUsingIncByLE(Std.Arithmetic.RippleCarryTTKIncByLE,_,_)",
  "Std.Arithmetic.IncByLUsingIncByLE(Std.Arithmetic.RippleCarryCGIncByLE,_,_)",
  "Std.Arithmetic.IncByLUsingIncByLE(QuantumArithmetic.JHHA2016.Add_Mod2N,_,_)",
  "Std.Arithmetic.IncByLUsingIncByLE(QuantumArithmetic.CDKM2004.Add,_,_)",
  "QuantumArithmetic.AdditionOrig.AddConstant",      
]

default_n_range = [3] + [int(round(2**(0.25*i))) for i in range(8,81)]
n_ranges = {op: default_n_range for op in ops}

metrics = ["Logical qubits", "Physical qubits", "Logical depth", "Runtime, seconds"]
charts = [{op: [] for op in ops} for _ in range(len(metrics))]

for op in ops:
    for n in n_ranges[op]:
        t0=time.time()
        estimates = json.loads(estimate_resources_constant_adder(op, n))
        #print(n, op, time.time()-t0, flush=True)
        charts[0][op].append(estimates['physicalCounts']['breakdown']['algorithmicLogicalQubits'])
        charts[1][op].append(estimates['physicalCounts']['physicalQubits'])
        charts[2][op].append(estimates['physicalCounts']['breakdown']['logicalDepth'])
        charts[3][op].append(estimates['physicalCounts']['runtime']/10**9)

fig, ax = plt.subplots(figsize=(15, 30), nrows=len(metrics), ncols=1, constrained_layout=True)
for i in range(len(metrics)):
    for op in ops:
        ax[i].plot(n_ranges[op], charts[i][op], label=op, marker='x')
    ax[i].set_xlim([2,max(max(n_ranges[op]) for op in ops)])
    ax[i].legend()
    ax[i].set_xlabel('Input size')
    ax[i].set_ylabel(metrics[i])
    ax[i].set_xscale('log')
    ax[i].set_yscale('log')
fig.suptitle("Resource estimation for constant adders", fontsize=16)
plt.show()